# Campsite Data Sync

Validates all `data/{agency}/WA/**/index.md` files, then rebuilds `data/campsites.json` as a full replacement.

**Run from project root:**
```bash
uv run --project data jupyter nbconvert --to notebook --inplace --execute data/sync_campsites.ipynb
```

In [1]:
import json
import re
from pathlib import Path

In [2]:
# Works whether invoked from project root or from data/
BASE = Path("data") if (Path("data") / "campsites.json").exists() else Path(".")
OUTPUT = BASE / "campsites.json"

AGENCIES = ["blm", "nps", "usfs", "wa-state-parks"]

VALID_TYPES = {"tent", "rv", "walk-in", "cabin", "bike-in", "parking"}

MONTH_MAP = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12,
}

WA_LAT = (45.5, 49.1)
WA_LNG = (-124.9, -116.9)

print(f"Base directory: {BASE.resolve()}")
print(f"Output: {OUTPUT.resolve()}")

Base directory: /Users/tommydoerr/dev/robot-geographical-society/data
Output: /Users/tommydoerr/dev/robot-geographical-society/data/campsites.json


## Output Location

Edit the path below before running Step 2. Defaults to `campsites.json` inside the data directory.

In [4]:
from ipyfilechooser import FileChooser

_fc = FileChooser(
    path=str(OUTPUT.parent.resolve()),
    filename=OUTPUT.name,
    title="Select output file:",
    show_hidden=False,
)

def _on_file_selected(chooser):
    global OUTPUT
    if chooser.selected:
        OUTPUT = Path(chooser.selected)

_fc.register_callback(_on_file_selected)
display(_fc)

FileChooser(path='/Users/tommydoerr/dev/intentcity/data', filename='campsites.json', title='Select output file…

## Step 1 — Parse & Validate

In [5]:
def parse_frontmatter(text: str) -> dict:
    """Parse YAML frontmatter + plain-text body from an index.md."""
    match = re.match(r"^---\n(.*?)\n---\n?(.*)", text, re.DOTALL)
    if not match:
        raise ValueError("No frontmatter block found")
    block, body = match.group(1), match.group(2).strip()

    result: dict = {}
    for line in block.splitlines():
        if ":" not in line:
            continue
        key, _, val = line.partition(":")
        key = key.strip()
        val = val.strip().split("#")[0].strip()  # strip inline comments

        if val.startswith("[") and val.endswith("]"):
            result[key] = [v.strip() for v in val[1:-1].split(",") if v.strip()]
        elif (val.startswith('"') and val.endswith('"')) or \
             (val.startswith("'") and val.endswith("'")):
            result[key] = val[1:-1]
        elif val == "true":
            result[key] = True
        elif val == "false":
            result[key] = False
        elif val == "null":
            result[key] = None
        else:
            try:
                result[key] = float(val) if "." in val else int(val)
            except ValueError:
                result[key] = val

    result["_notes"] = body or None
    return result

In [6]:
def validate(fm: dict, path: Path) -> list[str]:
    """Return a list of validation error strings (empty = valid)."""
    errors = []

    for field in ["name", "agency", "agency_short", "lat", "lng",
                  "sites", "types", "reservable", "year_round"]:
        if fm.get(field) is None:
            errors.append(f"missing required field: {field}")

    lat, lng = fm.get("lat"), fm.get("lng")
    if lat is not None and not (WA_LAT[0] <= lat <= WA_LAT[1]):
        errors.append(f"lat {lat} outside WA range {WA_LAT}")
    if lng is not None and not (WA_LNG[0] <= lng <= WA_LNG[1]):
        errors.append(f"lng {lng} outside WA range {WA_LNG}")

    open_date = fm.get("open_date")
    if fm.get("year_round") and open_date is not None:
        errors.append("year_round: true but open_date is set")
    if not fm.get("year_round") and open_date is not None:
        month_name = str(open_date).split()[0].lower()
        if month_name not in MONTH_MAP:
            errors.append(f"open_date month '{month_name}' not recognised")

    for t in fm.get("types") or []:
        if t not in VALID_TYPES:
            errors.append(f"unknown type '{t}' — valid: {sorted(VALID_TYPES)}")

    return errors

In [7]:
all_files = []
for agency in AGENCIES:
    agency_dir = BASE / "campsites" / agency / "WA"
    if agency_dir.is_dir():
        all_files.extend(sorted(agency_dir.rglob("index.md")))

print(f"Found {len(all_files)} campsite files\n")

validation_errors: dict[str, list[str]] = {}
parsed: dict[Path, dict] = {}

for path in all_files:
    try:
        fm = parse_frontmatter(path.read_text())
        errs = validate(fm, path)
        if errs:
            validation_errors[str(path)] = errs
        else:
            parsed[path] = fm
    except Exception as exc:
        validation_errors[str(path)] = [f"parse error: {exc}"]

if validation_errors:
    print(f"✗ {len(validation_errors)} file(s) with errors:\n")
    for p, errs in validation_errors.items():
        rel = Path(p).relative_to(Path.cwd()) if Path(p).is_absolute() else Path(p)
        print(f"  {rel}")
        for e in errs:
            print(f"    · {e}")
else:
    print(f"✓ All {len(all_files)} files passed validation")

Found 75 campsite files

✓ All 75 files passed validation


In [8]:
# Halt notebook execution if any file failed validation.
# Fix the errors above, then re-run.
if validation_errors:
    raise RuntimeError(
        f"{len(validation_errors)} validation error(s) — fix before syncing. "
        "See output above for details."
    )

## Step 2 — Build & Write GeoJSON

In [9]:
def open_date_to_month(val) -> int | None:
    if not val:
        return None
    return MONTH_MAP.get(str(val).strip().split()[0].lower())


def build_feature(fm: dict) -> dict:
    return {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [fm["lng"], fm["lat"]],
        },
        "properties": {
            "name":            fm["name"],
            "agency":          fm["agency"],
            "agency_short":    fm["agency_short"],
            "sites":           fm["sites"],
            "types":           fm["types"],
            "reservable":      fm["reservable"],
            "year_round":      fm["year_round"],
            "open_month":      open_date_to_month(fm.get("open_date")),
            "reservation_url": fm.get("reservation_url"),
            "notes":           fm.get("_notes"),
        },
    }


features = sorted(
    [build_feature(fm) for fm in parsed.values()],
    key=lambda f: (f["properties"]["agency_short"], f["properties"]["name"]),
)

geojson = {"type": "FeatureCollection", "features": features}
OUTPUT.write_text(json.dumps(geojson, indent=2) + "\n")

by_agency = {}
for f in features:
    a = f["properties"]["agency_short"]
    by_agency[a] = by_agency.get(a, 0) + 1

print(f"✓ {len(features)} campsites written to {OUTPUT}\n")
for agency, count in sorted(by_agency.items()):
    print(f"  {agency:20s} {count}")

✓ 75 campsites written to /Users/tommydoerr/dev/intentcity/data/campsites.json

  blm                  5
  nps                  24
  usfs                 21
  wa-state-parks       25
